# HYDRO30 Surface Water Extent Map: Process RTC stack

This notebook processes a stack of HYDRO30 products from a stack of HyP3 or OPERA dual-pol RTCs.

<img style= "padding: 7px" src="https://courses.edx.org/asset-v1:AlaskaX+SAR-401+3T2020+type@asset+block@Watermappingworkflow2.jpg" width="70%"/>

---

<div class="alert alert-info" style="display: flex; align-items: center; font-family: 'Times New Roman', Times, serif; background-color: #d1ecf1;">
  <div style="display: flex; align-items: center; width: 5%;">
    <a href="https://github.com/HydroSAR/HydroSAR/issues">
      <img src="https://opensarlab-docs.asf.alaska.edu/opensarlab-notebook-assets/logos/github_issues.png" alt="GitHub logo over the word Issues" style="width: 100px;">
    </a>
  </div>
  <div style="width: 95%;">
    <b>Did you find a bug? Do you have a feature request? Do you have questions about HydroSAR?</b>
    <br/>
    Explore GitHub Issues on this Jupyter Book's GitHub repository. Find solutions, add to the discussion, or start a new bug report or feature request: <a href="https://github.com/HydroSAR/HydroSAR/issues">HydroSAR Issues</a>
  </div>
</div>

<div class="alert alert-info" style="display: flex; align-items: center; justify-content: space-between; font-family: 'Times New Roman', Times, serif; background-color: #d1ecf1;">
  <div style="display: flex; align-items: center; margin-right: 10px; width: 5%;">
    <a href="mailto:uso@asf.alaska.edu">
      <img src="https://opensarlab-docs.asf.alaska.edu/opensarlab-notebook-assets/logos/ASF_support_logo.png" alt="ASF logo" style="width: 100px">
    </a>
  </div>
  <div style="width: 95%;">
    <b>Have a question related to SAR or ASF data access?</b>
    <br/>
    Contact ASF User Support: <a href="mailto:uso@asf.alaska.edu">uso@asf.alaska.edu</a>
  </div>
</div>

---
## 0. Select a directory holding an RTC stack prepared with the [Prepare a SAR RTC Data Stack for HydroSAR notebook](Prepare_HydroSAR_RTC_Stack.ipynb)

The directory should contain the following subdirectories:
- `VH`
- `VV`

In [1]:
from hydrosar.water_map import make_water_map
import inspect

print(inspect.getsource(make_water_map))

ERROR 1: PROJ: proj_create_from_database: Open of /home/jovyan/.local/envs/hydrosarng/share/proj failed


def make_water_map(out_raster: Union[str, Path], vv_raster: Union[str, Path], vh_raster: Union[str, Path],
                   hand_raster: Optional[Union[str, Path]] = None, tile_shape: Tuple[int, int] = (100, 100),
                   max_vv_threshold: float = -15.5, max_vh_threshold: float = -23.0,
                   hand_threshold: float = 15., hand_fraction: float = 0.8, membership_threshold: float = 0.45):
    """Creates a surface water extent map from a Sentinel-1 RTC product

    Create a surface water extent map from a dual-pol Sentinel-1 RTC product and
    a HAND image. The HAND image must be pixel-aligned (same extent and size) to
    the RTC images. The water extent maps are created using an adaptive Expectation
    Maximization thresholding approach and refined with Fuzzy Logic.

    The input images are broken into a set of corresponding tiles with a shape of
    `tile_shape`, and a set of tiles are selected from the VH RTC
    image that contain water boundaries to determ

In [2]:
from pathlib import Path

from ipyfilechooser import FileChooser

fc = FileChooser(Path.home())
display(fc)

FileChooser(path='/home/jovyan', filename='', title='', show_hidden=False, select_desc='Select', change_desc='…

In [3]:
data_dir = Path(fc.selected_path)
vh_dir = data_dir / 'RTC_GAMMA' #'VH'
vv_dir = data_dir / 'RTC_GAMMA' #'VV'

vh_paths = sorted(list(vh_dir.glob('*VH*.tif')))
vv_paths = sorted(list(vv_dir.glob('*VV*.tif')))

water_mask_dir = data_dir / 'Water_Masks'
water_mask_dir.mkdir(exist_ok=True)

In [4]:
vv_paths

[PosixPath('/home/jovyan/CRREL/OverflowDetection/Dataset/water_mask/TananaRiver/RTC_GAMMA/S1C_IW_20250610T032011_DVP_RTC10_G_gpufem_C3A2_VV_subset.tif'),
 PosixPath('/home/jovyan/CRREL/OverflowDetection/Dataset/water_mask/TananaRiver/RTC_GAMMA/S1C_IW_20250622T032012_DVP_RTC10_G_gpufem_ECF3_VV_subset.tif'),
 PosixPath('/home/jovyan/CRREL/OverflowDetection/Dataset/water_mask/TananaRiver/RTC_GAMMA/S1C_IW_20250704T032012_DVP_RTC10_G_gpufem_E2FE_VV_subset.tif'),
 PosixPath('/home/jovyan/CRREL/OverflowDetection/Dataset/water_mask/TananaRiver/RTC_GAMMA/S1C_IW_20250716T032013_DVP_RTC10_G_gpufem_AB83_VV_subset.tif'),
 PosixPath('/home/jovyan/CRREL/OverflowDetection/Dataset/water_mask/TananaRiver/RTC_GAMMA/S1C_IW_20250728T032014_DVP_RTC10_G_gpufem_43D1_VV_subset.tif'),
 PosixPath('/home/jovyan/CRREL/OverflowDetection/Dataset/water_mask/TananaRiver/RTC_GAMMA/S1C_IW_20250809T032014_DVP_RTC10_G_gpufem_322A_VV_subset.tif'),
 PosixPath('/home/jovyan/CRREL/OverflowDetection/Dataset/water_mask/TananaRi

In [5]:
import opensarlab_lib as asfn
def get_dates(product_paths):
    dates = []
    for pth in product_paths:
        dates.append(asfn.date_from_product_name(str(pth)).split('T')[0])
    return dates

In [6]:
from typing import Union
import os
from osgeo import gdal, osr
def get_epsg(geotiff_path: Union[str, os.PathLike]) -> str:
    """
    Takes: A string path or posix path to a GeoTiff

    Returns: The string EPSG of the Geotiff
    """
    ds = gdal.Open(str(geotiff_path))
    proj = ds.GetProjection()
    srs = osr.SpatialReference()
    srs.ImportFromWkt(proj)
    srs.AutoIdentifyEPSG()
    return srs.GetAuthorityCode(None)
    

In [7]:
from shapely.geometry import Polygon
import rasterio
def get_geotiff_bbox(geotiff_path: Union[str, os.PathLike], dst_epsg: str=None) -> Polygon:
    with rasterio.open(geotiff_path) as dataset:
        bounds = dataset.bounds
        min_x, min_y = (bounds.left, bounds.bottom)
        max_x, max_y = (bounds.right, bounds.top)
        
    if dst_epsg:
        srs_crs = dataset.crs
        transformer = Transformer.from_crs(srs_crs, f'EPSG:{str(dst_epsg)}', always_xy=True)
        min_x, min_y = transformer.transform(bounds.left, bounds.bottom)
        max_x, max_y = transformer.transform(bounds.right, bounds.top)

    return Polygon([
        (min_x, min_y),
        (max_x, min_y),
        (max_x, max_y),
        (min_x, max_y),
        (min_x, min_y)
    ])

In [8]:
import sys

import geopandas as gpd

current = Path("..").resolve()
sys.path.append(str(current))
# import util# import util.util as util

gdf = gpd.GeoDataFrame(
    {
    'file': vh_paths + vv_paths,
    'data_type': ['VV_RTC' if 'VV' in str(pth) else 'VH_RTC' for pth in vh_paths + vv_paths],
    'SAR_acquisition_date': get_dates(vh_paths + vv_paths),
    'EPSG': [get_epsg(p) for p in vh_paths + vv_paths],
    'geometry': [get_geotiff_bbox(p) for p in vh_paths + vv_paths],
    }
                  )
display(gdf)
if gdf['EPSG'].nunique() > 1:
    raise Exception(f'Error: Data is in {gdf["EPSG"].nunique()} projections.')

,file,data_type,SAR_acquisition_date,EPSG,geometry
0,/home/jovyan/CRREL/OverflowDetection/Dataset/w...,VH_RTC,20250610,32606,"POLYGON ((399490 7158710, 462640 7158710, 4626..."
1,/home/jovyan/CRREL/OverflowDetection/Dataset/w...,VH_RTC,20250622,32606,"POLYGON ((399490 7158710, 462640 7158710, 4626..."
2,/home/jovyan/CRREL/OverflowDetection/Dataset/w...,VH_RTC,20250704,32606,"POLYGON ((399490 7158710, 462640 7158710, 4626..."
3,/home/jovyan/CRREL/OverflowDetection/Dataset/w...,VH_RTC,20250716,32606,"POLYGON ((399490 7158710, 462640 7158710, 4626..."
4,/home/jovyan/CRREL/OverflowDetection/Dataset/w...,VH_RTC,20250728,32606,"POLYGON ((399490 7158710, 462640 7158710, 4626..."
5,/home/jovyan/CRREL/OverflowDetection/Dataset/w...,VH_RTC,20250809,32606,"POLYGON ((399490 7158710, 462640 7158710, 4626..."
6,/home/jovyan/CRREL/OverflowDetection/Dataset/w...,VH_RTC,20250821,32606,"POLYGON ((399490 7158710, 462640 7158710, 4626..."
7,/home/jovyan/CRREL/OverflowDetection/Dataset/w...,VV_RTC,20250610,32606,"POLYGON ((399490 7158710, 462640 7158710, 4626..."
8,/home/jovyan/CRREL/OverflowDetection/Dataset/w...,VV_RTC,20250622,32606,"POLYGON ((399490 7158710, 462640 7158710, 4626..."
9,/home/jovyan/CRREL/OverflowDetection/Dataset/w...,VV_RTC,20250704,32606,"POLYGON ((399490 7158710, 462640 7158710, 4626..."


---
## 1. Generate water extent maps

In [9]:
%%time
from datetime import datetime

from hydrosar.water_map import make_water_map
from tqdm.auto import tqdm

for d in tqdm(set(gdf['SAR_acquisition_date'])):
    vh_raster = gdf.loc[(gdf['SAR_acquisition_date'] == d) & (gdf['data_type'] == 'VH_RTC')]['file'].iloc[0]
    vv_raster = gdf.loc[(gdf['SAR_acquisition_date'] == d) & (gdf['data_type'] == 'VV_RTC')]['file'].iloc[0]

    d = datetime.strptime(d, "%Y%m%d")
    water_extent_raster = water_mask_dir / f'water_extent_{datetime.strftime(d, "%Y%m%d")}.tif'

    make_water_map(
        water_extent_raster, 
        vv_raster, 
        vh_raster, 
        tile_shape=(100, 100),
        max_vv_threshold=-15.5, 
        max_vh_threshold=-23., 
        hand_threshold=15., 
        hand_fraction=0.8
    )

  0%|          | 0/7 [00:00<?, ?it/s]

CPU times: user 2min 25s, sys: 11.7 s, total: 2min 36s
Wall time: 2min 24s


*HYDRO30_Stack_Processing.ipynb - Version 1.0.0 - May 2024*